# Colab classification of statistics descriptions 

## Set-up

In [ ]:
!pip install -U datasets
!pip install ray

In [ ]:
import datasets
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import random
import numpy as np
from datasets import Dataset

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Classification parameters
# Clear PyTorch GPU cache
torch.cuda.empty_cache()

# Training set parameters
fraction_positives = 1
fraction_mining = 0.1
ratio_factor_negatives = 1            # share of negatives: n_negatives = 1.5 * n_positives
fraction_positives_in_test = 0.05     # percent positives in test set
fraction_remaining_unlabeled = 1      # percent to keep of unlabeled for reconstructing training set

# Model parameters
model_id = "xlm-roberta-large"
max_token_length = None               # None uses max model allows

# Hyperparameter tuning
fraction_positives_for_tuning = 0.3
n_trials_ray = 30
n_cpu_ray = 2
n_gpu_ray = 1                         # For colab, 1 GPU max

In [ ]:
# Mount drive to import datasets
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Load the data
stat_to_mine = pd.read_feather("./drive/MyDrive/Colab Notebooks/stat_to_mine.feather")

# Discard any missing values in text_mining_description for robustness
stat_to_mine = stat_to_mine.dropna(subset=['text_mining_description'])

## Initial classification

This step was only done initially to create a first training set from `stat_to_mine` by stratified sampling and including minig descriptions as negatives. Later, this initial training set was adjusted manually by removing false positives and including desired descriptions (e.g. related to birth registration) to guide the model during training to capute a desired concept of data & statistics.

### Construct training and test set 

In [ ]:
# Filter out positive, mining and unlabeled samples
positive = stat_to_mine[stat_to_mine['is_statistics'] == True].copy()
mining = stat_to_mine[stat_to_mine['is_mining'] == True].copy()
unlabeled = stat_to_mine[(stat_to_mine['is_statistics'] == False) & (stat_to_mine['is_mining'] == False)].copy()

# Sample randomly fraction_positives for positives
positive = positive.sample(frac=fraction_positives, random_state=SEED)

In [ ]:
# --------------------------  Mining  ------------------------------------------
# Sample fraction_mining of mining (scale the number of mining samples with fraction_positives so that the ratio is maintained)
n_mining = int(fraction_positives * fraction_mining * len(mining))
mining_sampled = mining.sample(n=n_mining, random_state=SEED)

# ---------------- Stratified sampling from unlabeled --------------------------
# Stratification acc. to language and length of text_mining_description
# Create length bins according to language and length of text_mining_description
positive.loc[:, 'length_bin'] = pd.cut(positive['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True)
unlabeled.loc[:, 'length_bin'] = pd.cut(unlabeled['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True)
mining_sampled.loc[:, 'length_bin'] = pd.cut(mining_sampled['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True)

# Create joint stratum
positive.loc[:, 'stratum'] = positive['language'].astype(str) + "_" + positive['length_bin'].astype(str)
unlabeled.loc[:, 'stratum'] = unlabeled['language'].astype(str) + "_" + unlabeled['length_bin'].astype(str)
mining_sampled.loc[:, 'stratum'] = mining_sampled['language'].astype(str) + "_" + mining_sampled['length_bin'].astype(str)

# Calculate stratum distribution in positives
stratum_counts = positive['stratum'].value_counts(normalize=True)

# Sample from unlabeled according to this distribution
n_negatives = ratio_factor_negatives*len(positive)-n_mining
samples_per_stratum = (stratum_counts * n_negatives).round().astype(int)

# Sample from unlabeled
neg_samples = []
for stratum, n in samples_per_stratum.items():
    candidates = unlabeled[unlabeled['stratum'] == stratum]
    if len(candidates) >= n:
        neg_samples.append(candidates.sample(n=n, random_state=SEED))
    else:
        neg_samples.append(candidates)  # take all if not enough

unlabeled_sampled = pd.concat(neg_samples).sample(frac=1, random_state=SEED)  # shuffle

# Combine positives and sampled negatives
balanced = pd.concat([positive, unlabeled_sampled, mining_sampled]).reset_index(drop=True)

# Keep the remaining unlabeled samples for later
remaining_unlabeled = unlabeled.drop(unlabeled_sampled.index)

In [ ]:
# Split into train & test sets (80% train, 20% test)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    balanced['text_mining_description'],
    balanced['is_statistics'].astype(int),
    test_size=0.1,
    stratify=balanced['is_statistics'],
    random_state=SEED
)

balanced['is_statistics'].value_counts()

Optional: unbalance test set to resemble real world distribution of stat projects in CRS, but was later abbandoned in favor of a manual inspection of descriptions around the eventual threshold

In [ ]:
# Unbalance test set to resemble real world distribution
n_pos_test = sum(test_labels)
n_neg_test = len(test_labels) - n_pos_test
n_neg_add = int((n_pos_test - fraction_positives_in_test*(n_pos_test + n_neg_test))/fraction_positives_in_test)

print(n_neg_add)

additional_unlabeled = remaining_unlabeled.sample(n=n_neg_add, random_state=SEED)

additional_unlabeled_texts  = additional_unlabeled['text_mining_description']
additional_unlabeled_labels = additional_unlabeled['is_statistics'].astype(int)

additional_unlabeled_labels.value_counts()

In [ ]:
test_texts = pd.concat([test_texts, additional_unlabeled_texts])
test_labels =  pd.concat([test_labels, additional_unlabeled_labels])

print(f"Final‑test prevalence: {np.mean(test_labels)}")

### Tokenization

In [ ]:
# Load tokenizer and tokenize data
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

# Tokenize dataset with custom tokenize()
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

### Model training

In [ ]:
# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./drive/MyDrive/Colab Notebooks/models/results",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=1e-5,
    logging_dir="./drive/MyDrive/Colab Notebooks/models/logs",
    eval_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    save_total_limit=1, # keeps only the latest checkpoint
    metric_for_best_model="precision",
    fp16=True,
    report_to="none",
    seed=SEED
)

# Define metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

In [ ]:
predictions_output = trainer.predict(test_dataset)

In [ ]:
# Output test set with, text, label, and predicted probabilities
test_results = pd.DataFrame({
    'text': test_texts,
    'label': test_labels,
    'probability_is_statistics': torch.softmax(torch.tensor(predictions_output.predictions), dim=1)[:, 1].numpy()
})

#test_results.to_excel("./drive/MyDrive/Colab Notebooks/test_set_predicted.xlsx", index=False)

### Model evaluation

In [ ]:
# Interactive ROC curve
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score
import plotly.graph_objs as go
import matplotlib.pyplot as plt

# Extract labels and predicted probabilities
y_true = test_results['label'].values
y_scores = test_results['probability_is_statistics'].values

# Calculate ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = roc_auc_score(y_true, y_scores)

# Build interactive ROC plot
hover_text = [f"Threshold: {thr:.3f}<br>FPR: {f:.3f}<br>TPR: {t:.3f}"
              for thr, f, t in zip(thresholds, fpr, tpr)]

trace = go.Scatter(
    x=fpr,
    y=tpr,
    mode='lines+markers',
    text=hover_text,
    hoverinfo='text',
    name=f'ROC curve (AUC = {roc_auc:.3f})',
    line=dict(color='orange')
)

line_random = go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random Guessing',
    line=dict(dash='dash', color='navy')
)

layout = go.Layout(
    title='ROC Curve',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate (Recall)'),
    legend=dict(x=0.6, y=0.05),
    hovermode='closest'
)

fig = go.Figure(data=[trace, line_random], layout=layout)
fig.show()


In [ ]:
# Precision recall curve
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, pr_thresholds = precision_recall_curve(y_true, y_scores)
pr_auc = average_precision_score(y_true, y_scores)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR curve (AP = {pr_auc:.3f})', color='green', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curve on Unbalanced Test Set')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

## Rebuild training set with reliable negatives

### Predict unlabeled

Predict the part of the unlabeled set to obtain reliable negatives for reconstructing the training set later on

In [ ]:
# Sample from remaining unlabeled
remaining_unlabeled = remaining_unlabeled.sample(frac=fraction_remaining_unlabeled, random_state=SEED)

# Construct dataset 
remaining_unlabeled_texts = remaining_unlabeled["text_mining_description"].tolist()
remaining_unlabeled_dataset = Dataset.from_dict({'text': remaining_unlabeled_texts})
remaining_unlabeled_dataset = remaining_unlabeled_dataset.map(tokenize, batched=True)

In [ ]:
# Predict the unlabeled samples
pred_output = trainer.predict(remaining_unlabeled_dataset)
logits = pred_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
remaining_unlabeled["probability_is_statistics"] = probs

In [ ]:
# Save to xlsx to inspect manually
remaining_unlabeled.to_excel("./drive/MyDrive/Colab Notebooks/unlabeled_predictions_1training_large.xlsx", index=False)

### Rebuild training set with reliable negatives

In [ ]:
# --------------------------- Positives ----------------------------------------
# Filter out positive and unlabeled samples
positive = stat_to_mine[stat_to_mine['is_statistics'] == True].copy()

# Sample randomly fractional positives for tuning - use small training set to reduce training time as it is only used for comparing hyperparameter configurations
positive = positive.sample(frac=fraction_positives_for_tuning, random_state=SEED)

# Use the same stratum distribution as positives
positive.loc[:, 'length_bin'] = pd.cut(
    positive['text_mining_description'].str.len(), bins=6, labels=False, include_lowest=True
)
positive.loc[:, 'stratum'] = positive['language'].astype(str) + "_" + positive['length_bin'].astype(str)
stratum_counts = positive['stratum'].value_counts(normalize=True)

# ------------------------------- Mining ---------------------------------------
mining = stat_to_mine[stat_to_mine['is_mining'] == True].copy()
n_mining = int(fraction_positives_for_tuning * fraction_mining * len(mining))
mining_sampled = mining.sample(n=n_mining, random_state=SEED)


# ------------------------- Reliable negatives ---------------------------------
# Select reliable negatives: probability < 0.8 (stratum & language still present from earlier), 0.8 chosen to ensure that we capture a wide range of examples so that some near statistics projects are included 
reliable_negatives = remaining_unlabeled[remaining_unlabeled['probability_is_statistics'] < 0.8].copy()

n_negatives = len(positive)-n_mining
samples_per_stratum = (stratum_counts * n_negatives).round().astype(int)

neg_samples = []
for stratum, n in samples_per_stratum.items():
    candidates = reliable_negatives[reliable_negatives['stratum'] == stratum]
    if len(candidates) >= n:
        neg_samples.append(candidates.sample(n=n, random_state=SEED))
    else:
        neg_samples.append(candidates)

reliable_negatives_sampled = pd.concat(neg_samples).sample(frac=1, random_state=SEED)
reliable_negatives_sampled = reliable_negatives_sampled.drop(columns=['probability_is_statistics', 'length_bin', 'stratum'])

# Rebuild the training set
positive = positive.drop(columns=['length_bin', 'stratum'])
training_dataset_with_reliables = pd.concat([positive, mining_sampled, reliable_negatives_sampled]).reset_index(drop=True)
training_dataset_with_reliables = training_dataset_with_reliables.drop(columns=['is_mining'], errors='ignore')


training_dataset_with_reliables['is_statistics'].value_counts()

In [ ]:
# Split into train+val and test sets first (80% train+val, 20% test)
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    training_dataset_with_reliables['text_mining_description'],
    training_dataset_with_reliables['is_statistics'].astype(int),
    test_size=0.1,
    stratify=training_dataset_with_reliables['is_statistics'],
    random_state=SEED
)

# Now split train+val into train and validation sets (75% train, 25% val of the 80%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=1/9,  # 1/9 x 0.9 = 0.1, so final split is 80% train, 10% val, 10% test
    stratify=train_val_labels,
    random_state=SEED
)

Optional: unbalance test set to resemble real world distribution of stat projects in CRS, but was later abbandoned in favor of a manual inspection of descriptions around the eventual threshold

In [ ]:
# Augment validation set and test set with true negatives from reliable_negatives
n_pos_val = sum(val_labels)
n_neg_val = len(val_labels) - n_pos_val
n_neg_add_hyper_tuning = int((n_pos_val - fraction_positives_in_test*(n_pos_val + n_neg_val))/fraction_positives_in_test)

# Sample
additional_negatives_val = remaining_unlabeled.sample(n=n_neg_add_hyper_tuning, random_state=SEED)
additional_negatives_test = remaining_unlabeled.sample(n=n_neg_add_hyper_tuning, random_state=SEED)

# Get
val_texts = pd.concat([val_texts, additional_negatives_val['text_mining_description']])
val_labels =  pd.concat([val_labels, additional_negatives_val['is_statistics'].astype(int)])

test_texts = pd.concat([test_texts, additional_negatives_test['text_mining_description']])
test_labels =  pd.concat([test_labels, additional_negatives_test['is_statistics'].astype(int)])

print(f"Final‑val/test hyper tuning prevalence: {np.mean(val_labels)}/{np.mean(test_labels)}")
print(f"Number of positives in val/test set: {sum(val_labels)}/{sum(test_labels)}")

In [ ]:
# Load tokenizer and tokenize data
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
val_dataset = Dataset.from_dict({'text': val_texts.tolist(), 'label': val_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

## Hyperparameter tuning 

In [ ]:
import os
import ray
from ray import tune
from ray.tune import JupyterNotebookReporter
from ray.tune.schedulers import ASHAScheduler
from transformers import TrainerCallback

reporter = JupyterNotebookReporter(
    parameter_columns=["learning_rate", "per_device_train_batch_size"],
    metric_columns=["eval_accuracy", "eval_f1", "eval_auc", "eval_loss", "epoch"]
)

scheduler = ASHAScheduler(
        metric="accuracy", #loss defined in training function
        mode="max",
        grace_period=1, # iterations after which ASHA terminates if metric doesn't improve
        reduction_factor=10) # iterations after which ASHA reevaluates metric and determine to stop a trial

# Define metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = np.argmax(logits, axis=-1)

    # Get probability for positive class (class 1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

    # Standard classification metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)

    # Compute AUC (requires probabilities, not argmax predictions)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        # In case only one class is present in labels
        auc = float('nan')

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "auc": auc
    }

class TuneReportCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            # Report all available metrics (e.g., loss, accuracy, etc.)
            tune.report(metrics)

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

config = {
    "learning_rate": tune.uniform(1e-5, 5e-5),
    "weight_decay":  tune.uniform(0.0, 0.3),
    "num_train_epochs": tune.choice([2, 3, 4]),
    "per_device_train_batch_size": tune.choice([8, 16, 32]),
    "warmup_ratio": tune.choice([0.0, 0.06, 0.1]),
    "gradient_accumulation_steps": tune.choice([1, 2, 4]),
    "adam_beta1": tune.choice([0.9, 0.95]),
    "adam_beta2": tune.choice([0.98, 0.999]),
}

train_ref = ray.put(train_dataset)
val_ref = ray.put(val_dataset)

def trainer_init(config):
    output_dir = os.path.abspath("./models/results")

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["per_device_train_batch_size"],
        gradient_accumulation_steps = config["gradient_accumulation_steps"],
        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],
        adam_beta1=config["adam_beta1"],
        adam_beta2=config["adam_beta2"],
        per_device_eval_batch_size=8,
        logging_dir="./models/logs",
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        metric_for_best_model="auc",
        report_to="none",
        seed=SEED
    )

    return Trainer(
        args=args,
        train_dataset=ray.get(train_ref),
        eval_dataset=ray.get(val_ref),
        compute_metrics=compute_metrics,
        model_init=model_init,
        callbacks=[TuneReportCallback()]
    )


def run_ray_train(config):
    trainer = trainer_init(config)
    trainer.train()
    eval_metrics = trainer.evaluate()
    tune.report(eval_metrics)

In [ ]:
storage_path = os.path.abspath("./drive/MyDrive/Colab Notebooks/ray_tune_results")

# Hyperparameter tuning with Ray Tune
analysis = tune.run(
    run_ray_train,
    config=config,
    resources_per_trial={"cpu": n_cpu_ray, "gpu": n_gpu_ray},
    metric="eval_accuracy",
    mode="max",
    num_samples=n_trials_ray,
    #scheduler=scheduler,
    progress_reporter=reporter,
    storage_path=storage_path,
    name="tune_xlm_roberta"
)

In [ ]:
# Best config from ray results - one for xlm-roberta-base and one for xlm-roberta-large
best_config_base = {
  "adam_beta1": 0.95,
  "adam_beta2": 0.98,
  "gradient_accumulation_steps": 2,
  "learning_rate": 3.5175945525410505e-05,
  "num_train_epochs": 4,
  "per_device_train_batch_size": 8,
  "warmup_ratio": 0.0,
  "weight_decay": 0.20872460669538515
}

best_config_large = {
    "learning_rate": 2.5642424302929634e-05,
    "weight_decay": 0.0546708263364187,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 16,
    "warmup_ratio": 0.06,
    "gradient_accumulation_steps": 2,
    "adam_beta1": 0.95,
    "adam_beta2": 0.999
    }

## Final classification of full unlabeled set

### Retraining with best config from hyperparameter search

In [ ]:
# Load manually corrected full training set
balanced = pd.read_excel("./drive/MyDrive/Colab Notebooks/training_set_manually_corrected.xlsx")

balanced.drop(columns=['is_statistics', 'probability_is_statistics'], inplace=True)

# Rename 'manually_corrected' to 'is_statistics'
balanced.rename(columns={'manually_corrected': 'is_statistics'}, inplace=True)

In [ ]:
# Split into train+val and test sets first (80% train+val, 20% test)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    balanced['text_mining_description'],
    balanced['is_statistics'].astype(int),
    test_size=0.1,
    stratify=balanced['is_statistics'],
    random_state=SEED
)

In [ ]:
# Load tokenizer and tokenize data
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'label': train_labels.tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'label': test_labels.tolist()})

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

best_config = best_config_large.copy()

# Define training arguments
training_args = TrainingArguments(
    output_dir="./drive/MyDrive/Colab Notebooks/models/results",
    num_train_epochs=best_config["num_train_epochs"],
    per_device_train_batch_size=best_config["per_device_train_batch_size"],
    per_device_eval_batch_size=64,
    gradient_accumulation_steps = best_config["gradient_accumulation_steps"],
    warmup_ratio=best_config["warmup_ratio"],
    weight_decay=best_config["weight_decay"],
    learning_rate=best_config["learning_rate"],
    adam_beta1=best_config["adam_beta1"],
    adam_beta2=best_config["adam_beta2"],
    logging_dir="./drive/MyDrive/Colab Notebooks/models/logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1, # keeps only the latest checkpoint
    metric_for_best_model="precision",
    fp16=True,
    report_to="none",
    seed=SEED
)

# Define metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = np.argmax(logits, axis=-1)

    # Get probability for positive class (class 1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

    # Standard classification metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)

    # Compute AUC (requires probabilities, not argmax predictions)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        # In case only one class is present in labels
        auc = float('nan')

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "auc": auc
    }

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

In [ ]:
# Evaluate or resume training
predictions_output = trainer.predict(test_dataset)

# Output test set with, text, label, and predicted probabilities
test_results = pd.DataFrame({
    'text': test_texts,
    'label': test_labels,
    'probability_is_statistics': torch.softmax(torch.tensor(predictions_output.predictions), dim=1)[:, 1].numpy()
})

Evaluation of final model: if desired, copy evaluation section from above and inspect interactive ROC

### Prediction of full unlabeled set

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=max_token_length)

In [ ]:
# Load unlabeled set
stat_to_mine = pd.read_feather("./drive/MyDrive/Colab Notebooks/stat_to_mine.feather")
unlabeled = stat_to_mine[(stat_to_mine['is_statistics'] == False) & (stat_to_mine['is_mining'] == False)].copy()

# Prepare unlabeled dataset
unlabeled_texts = unlabeled["text_mining_description"].tolist()
unlabeled_dataset = Dataset.from_dict({'text': unlabeled_texts})
unlabeled_dataset = unlabeled_dataset.map(tokenize, batched=True)

In [ ]:
# Load conflicting descriptions from conflicting_descr_stats
conflicting_stats = pd.read_feather("./drive/MyDrive/Colab Notebooks/conflicting_descr_stat.feather")

# Prepare conflicting dataset
conflicting_stats_texts = conflicting_stats["text_mining_description"].tolist()
conflicting_stats_dataset = Dataset.from_dict({'text': conflicting_stats_texts})
conflicting_stats_dataset = conflicting_stats_dataset.map(tokenize, batched=True)

If trainer no longer in memory, load from model checkpoint created during section "Retraining with best config from hyperparameter search". If it is still loaded, directly predict the unlabled set with the cell beneath.

In [ ]:
# Load best model from checkpoint - adjust checkpoint path to the one retraining
model_checkpoint = "./drive/MyDrive/Colab Notebooks/models/results/large-checkpoint-2169"  # Replace with the last saved checkpoint

# Load model from checkpoint
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint)

args = TrainingArguments(
    output_dir="./results",             # dummy
    per_device_eval_batch_size=128,
    do_train=False,
    do_eval=False,
    fp16=True,
    report_to="none",
    logging_strategy="no",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=args
)

#### Predict unlabeled

For entire unlabeled dataset, ~ 1.5h on A100 with xlm-roberta-large

In [ ]:
pred_output = trainer.predict(unlabeled_dataset)

# Probabilities
logits = pred_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

# Assign it to the new column
unlabeled['probability_is_statistics'] = probs

In [ ]:
# Save to feather
unlabeled.to_feather("./drive/MyDrive/Colab Notebooks/unlabeled_predicted_large.feather")

#### Predict conflicting descriptions

In [ ]:
pred_output_conflicting = trainer.predict(conflicting_stats_dataset)

# Probabilities
logits_conflicting = pred_output_conflicting.predictions
probs_conflicting = torch.softmax(torch.tensor(logits_conflicting), dim=1)[:, 1].numpy()

conflicting_stats['probability_is_statistics'] = probs_conflicting

conflicting_stats.to_feather("./drive/MyDrive/Colab Notebooks/conflicting_stat_predicted_large.feather")